In [ ]:
#@title Install dependencies (~35 s)
import os, time, glob, shutil, sys, subprocess
_T0 = time.time()

model = "openbind0" #@param ["openbind0", "openfold3", "boltz2", "protenix2", "rosettafold3", "chai1", "intellifold2", "opendde", "esmfold2", "esmfold2_lm600m", "esmfold2_lm300m", "alphafold3", "af2_ptm", "af2_multimer"]

persist_cache_to_drive = False #@param {type:"boolean"}
#@markdown - **persist_cache_to_drive**: keep the compiled model AND its lowering in
#@markdown   Drive so the next session skips both. Measured on a Colab T4 (58
#@markdown   residues): 72 s for the first fold, 26 s once the cache is there. The
#@markdown   cache populates itself -- nothing is downloaded -- so the saving starts
#@markdown   with your second fold. Never changes a result.

# Set any form field from the environment, for runs outside Colab:
#   AF3_NB_OVERRIDES='{"model": "boltz2"}'
import json as _json
for _k, _v in _json.loads(os.environ.get('AF3_NB_OVERRIDES', '{}')).items():
  if _k in globals():
    globals()[_k] = _v
    print(f'override: {_k} = {_v!r}')

VERSION = '3.1.11'          # the wheel, from PyPI
# Where the PYTHON comes from. A tag (`v3.1.11`) is the shipping notebook; the
# `colab` branch carries the experimental live-animation and steering code,
# which no release has. See the overlay below.
SOURCE = 'colab'
NATIVE_DIR = 'af3_native_weights'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'
AF2_DIR = 'af2_params'
IS_AF3 = (model == 'alphafold3')
IS_AF2 = model.startswith('af2_')
# int8 weights, expanded on load; AF2 and AF3 ship their own float32 files.
PRECISION = 'fp32' if (IS_AF3 or IS_AF2) else 'int8'


def _sh(cmd, what):
  """Run a shell command, raising if it fails."""
  if os.system(cmd) != 0:
    raise RuntimeError(f'{what} failed. The output is above.')


if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  # Installed with --no-deps, so the package's own imports are listed here.
  # Letting pip resolve them would re-download jax and the CUDA stack.
  # tokamax 0.0.11 WORKS ON jax 0.11.1 AND BREAKS ON 0.11.2, where
  # `jax.experimental.hijax.HiPrimitive` is gone -- and with tokamax
  # unimportable, nothing in the stack loads. A GPU image already has 0.11.1,
  # satisfies tokamax's `jax>=0.9.1`, and pip leaves it alone; a TPU image has
  # 0.7.2, FAILS that requirement, and pip upgrades it to the latest.
  # So LOOK BEFORE INSTALLING. Re-pinning a jax that is already right costs
  # 38 s and a libtpu download for nothing, and on a GPU image touching jax at
  # all would drag in the CUDA stack -- which is the very thing --no-deps is
  # here to avoid.
  import glob as _glob
  import importlib.metadata as _md
  try:
    _jax_now = _md.version('jax')
  except Exception:
    _jax_now = None
  _is_tpu = bool(_glob.glob('/dev/accel*') or os.environ.get('TPU_ACCELERATOR_TYPE'))
  if _jax_now == '0.11.1':
    pass                     # every GPU image today: nothing to do
  elif _is_tpu:
    print(f'jax {_jax_now} on a TPU runtime; pinning to 0.11.1 for tokamax')
    _sh('pip install -q "jax[tpu]==0.11.1"', 'pinning jax for the TPU runtime')
  else:
    # Deliberately NOT fixed here: jax[cuda12] would re-download the CUDA
    # stack. Say it plainly instead -- if the Colab GPU image ever moves to
    # 0.11.2, this line is what explains the import errors that follow.
    print(f'NOTE: jax {_jax_now} is not the 0.11.1 tokamax 0.0.11 expects; '
          'if imports fail with hijax.HiPrimitive, pin jax to 0.11.1.')
  # A PRE-AMPERE CARD HAS NO OTHER FUSED ATTENTION. cuDNN's SDPA wants SM80,
  # tokamax has no kernel for it, and XLA gates Pallas/Triton at sm_80 -- so a
  # T4, the commonest Colab GPU, runs the materialising XLA path for the 77%
  # of a pairformer pass that is triangle attention. Milot Mirdita's
  # colabfold-legacy-kernels is the exception: 3.0-3.35x that path on a T4,
  # measured at this model's own shape. Linux x86_64 wheels, and only where
  # the card can use them.
  import platform as _pyplat
  _cc0 = None
  if _pyplat.system() == 'Linux' and _pyplat.machine() == 'x86_64':
    try:
      _out0 = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap',
                              '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=15).stdout
      _cc0 = min(float(x) for x in _out0.split() if x.strip())
    except Exception:
      _cc0 = None
  if _cc0 is not None and _cc0 < 8.0:
    _sh('pip install -q colabfold-legacy-kernels==0.2.0',
        'installing the pre-Ampere fused kernels')
  elif _cc0 is not None:
    # ... and its sibling for Ampere and newer. Pure Python (Pallas), so this
    # is a small wheel and no build. It matters most on the L4 Colab offers:
    # tokamax's Triton refuses an Ada card outright ('Not supported on NVIDIA
    # A10') and cuDNN is what we fell back to -- this measured 3.3x cuDNN and
    # 11.9x XLA on an A10 at this model's triangle-attention shape.
    _sh('pip install -q colabfold-kernels==0.1.0',
        'installing the Ampere+ fused kernels')
  # zstandard IS one of them -- params.py, post_processing.py and
  # folding_input.py all import it. It happened to be preinstalled on the
  # GPU images, so its absence only showed up on a TPU runtime, as
  # `ModuleNotFoundError: No module named zstandard` from inside the fold,
  # long after the install cell had reported success.
  _sh("pip install -q dm-haiku==0.0.17 rdkit==2025.9.4 "
      "tokamax==0.0.11 ml_collections zstandard", 'installing dependencies')
  _sh("pip install -q git+https://github.com/sokrypton/py2Dmol.git",  # wheel lags the repo
      'installing py2Dmol')
  if IS_AF2:
    os.system("apt-get -qq install -y aria2 > /dev/null 2>&1")  # AF2's tar is 5.3 GB
  # Retried: PyPI's index can lag a just-published release by a few minutes.
  for _try in range(4):
    if os.system(f'pip install -q --no-deps alphafold3-colabfold=={VERSION}') == 0:
      break
    print(f'pip could not find {VERSION} yet; retrying in 20 s')
    time.sleep(20)
  else:
    raise RuntimeError(f'could not install alphafold3-colabfold=={VERSION}')

  # haiku 0.0.17 still calls the moved jax.core.DropVar.
  os.system("sed -i 's/jax.core.DropVar/jax.extend.core.DropVar/g' /usr/local/lib/python*/dist-packages/haiku/_src/jaxpr_info.py")
  import alphafold3  # confirms the install before anything depends on it
  os.system('touch ALPHAFOLD3_READY')
  print(f'Packages installed ({alphafold3.__file__}).')

# THE BRANCH OVERLAY RUNS EVERY TIME, not only on a fresh install. It used to
# sit inside the `ALPHAFOLD3_READY` guard, so a session that had already
# installed never refreshed it -- and a file ADDED on the branch (staged.py)
# could never arrive at all: `ImportError: cannot import name staged`, on a
# notebook whose overlay list had already been fixed. Five wgets, so there is
# no reason to skip them.
# EXPERIMENTAL BRANCH OVERLAY. The live cell needs library changes that are
# not in any release, so the branch's Python is copied over the installed
# wheel. That is legitimate here and nowhere else: `colab` changes no C++, so
# the compiled extension in the wheel is exactly the one this source expects.
# The moment a .cc changes on the branch this stops being true, which is what
# the assertion below is for -- and then the install becomes
#     pip install git+https://github.com/sokrypton/alphafold3@colab
# which builds the extension from scratch (~10 minutes on Colab).
# run_alphafold.py is a top-level script, not part of the package, and it is
# fetched HERE rather than under the install guard for the same reason as the
# rest of the overlay: on a warm session the guard is skipped and a stale copy
# would survive a push.
# run_alphafold.py COMES OUT OF THE SAME TARBALL as the package, below.
#
# It used to be its own wget from raw.githubusercontent, and that host serves a
# stale blob for a long time after a push -- a cache-buster query string did
# NOT help (verified: a fresh wget on the VM returned a file missing the
# newest commit while the GitHub API reported that commit as the branch head).
# Two failures came of it, and both looked like code bugs: the fold died on a
# validation the branch no longer had, quoting an error message that no longer
# existed. It is also the same shape as the overlay-list bug -- the script and
# the package coming from different places and disagreeing. One snapshot, one
# fetch, no way for them to drift.

# ONE TARBALL, for the script and (on a branch) the package. A tag needs
# run_alphafold.py too -- it is not part of the wheel -- so this runs either
# way and only the package overlay is conditional.
import importlib.metadata as _md
_root = os.path.dirname(_md.distribution('alphafold3-colabfold')
                        .locate_file('alphafold3'))
_ref = ('refs/heads/' + SOURCE if SOURCE != f'v{VERSION}'
        else 'refs/tags/' + SOURCE)
_sh(f'wget -q -O branch.tar.gz https://codeload.github.com'
    f'/sokrypton/alphafold3/tar.gz/{_ref}?nocache={int(time.time())}',
    f'fetching {SOURCE}')
_sh('rm -rf branch_src && mkdir branch_src && '
    'tar xzf branch.tar.gz -C branch_src --strip-components=1', 'unpacking it')
_sh('cp branch_src/run_alphafold.py run_alphafold.py',
    'taking run_alphafold.py from the tarball')

if SOURCE != f'v{VERSION}':
  # THE WHOLE TREE, not a list of files. There WAS a list, fetched from the
  # branch so it could not go stale against the branch -- and it went stale
  # against the WHEEL instead, which is the comparison that actually matters:
  # `git diff main colab` is empty for a file that main changed AFTER the
  # release was cut, so evoformer.py stayed at 3.1.11 while run_alphafold.py
  # came from the branch and asked it for bfloat16='intermediate'. The wheel's
  # assert had never heard of that value and every fold died in the first
  # recycle. A tarball of the branch has no list to keep in step: 8 MB, about
  # a second, one request instead of six.
  # `.` copies the CONTENTS, merging into the installed package, so the
  # compiled extension (.so) stays where the wheel put it -- which is the
  # whole reason overlaying source on a wheel is legitimate here.
  _sh(f'cp -r branch_src/src/alphafold3/. {_root}/alphafold3/',
      'overlaying the package')
  _sh('cp branch_src/dev/live/live_frames.py live_frames.py',
      'fetching live_frames.py')
  print(f'overlaid the {SOURCE} branch onto the {VERSION} wheel')

if SOURCE != f'v{VERSION}':
  # Proves the overlay landed rather than trusting the download. Checked by
  # READING the files: importing run_alphafold here would pull in
  # alphafold3.constants, whose pickles the input cell has not written yet
  # (FileNotFoundError: chemical_component_sets.pickle), and the failure
  # would land before CACHE_DIR is even defined.
  import importlib.metadata as _md2
  _r = os.path.dirname(_md2.distribution('alphafold3-colabfold')
                       .locate_file('alphafold3'))
  for _f, _needle in (
      ('run_alphafold.py', 'def live_model'),
      (os.path.join(_r, 'alphafold3/model/model.py'), "stage='all'"),
      (os.path.join(_r, 'alphafold3/model/staged.py'), 'def make_stages'),
  ):
    assert _needle in open(_f).read(), (
        f'{_f} is not the {SOURCE} version ({_needle!r} missing) -- the '
        'overlay did not take effect and the live cell would fail')

# tokamax's Triton kernels need more shared memory than Ada cards have, so
# restrict them to datacenter GPUs (A100 cc 8.0, H100 cc 9.0+).
#
# DO NOT IMPORT tokamax TO DO IT, AND DO NOT DO IT OFF A GPU. `import tokamax`
# imports jax, and on a TPU runtime the first process to touch jax OWNS the
# chip -- so this line, whose only job is to edit a CUDA policy, took the TPU
# and left the fold's own subprocess with
#     ABORTED: The TPU is already in use by process with pid <the kernel>
# and a silent fall back to "CPU-only inference". find_spec locates the file
# without executing the package, and the patch is skipped where it means
# nothing. (platform.detect_device() is jax-free by design -- checked.)
from alphafold3.model.components import platform as _plat
_dev0, _cap0 = _plat.detect_device()
if _dev0 != 'gpu' or not _plat.needs_tokamax_patch(_cap0):
  print(f'tokamax patch not needed on this device ({_dev0}, cc {_cap0}).')
try:
  import importlib.util as _ilu
  if _dev0 != 'gpu' or not _plat.needs_tokamax_patch(_cap0):
    raise SystemExit  # caught below; nothing to patch
  _spec = _ilu.find_spec('tokamax')
  _gu = os.path.join(os.path.dirname(_spec.origin), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except SystemExit:
  pass
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights, fetched in the background by the same code the run uses.
STAMP = f'WEIGHTS_DONE_{model}_{PRECISION}'
if IS_AF3 and not os.path.isfile(STAMP):
  # DeepMind's own release, subject to the AlphaFold 3 terms of use, which
  # run_alphafold prints at startup. A copy you already have in NATIVE_DIR is
  # used as-is.
  os.makedirs(NATIVE_DIR, exist_ok=True)
  if glob.glob(f'{NATIVE_DIR}/*.bin.zst'):
    open(STAMP, 'w').close()
    print(f'Using the AlphaFold 3 parameters already in {NATIVE_DIR}/.')
  else:
    print('Downloading AlphaFold 3 parameters (~1 GB)...')
    os.system(f'(wget -O {NATIVE_DIR}/af3.bin.zst "{AF3_WEIGHTS_URL}"'
              f' > {STAMP}.log 2>&1 && touch {STAMP}) &')
elif not (IS_AF3 or os.path.isfile(STAMP)):
  _script, _args = ('prefetch_af2.py', AF2_DIR) if IS_AF2 else (
      'prefetch_weights.py', f'{model} {PRECISION}')
  print(f'Downloading {"official AlphaFold 2 parameters (CC BY 4.0)" if IS_AF2 else model} weights...')
  with open(_script, 'w') as fh:
    fh.write('import sys\n'
             'from alphafold3.model import weights\n'
             + ('print(weights.ensure_af2_params(sys.argv[1]))\n' if IS_AF2 else
                'print(weights.ensure_weights(sys.argv[1], None, precision=sys.argv[2]))\n'))
  os.system(f'(python {_script} {_args} > {STAMP}.log 2>&1 && touch {STAMP}) &')

# /tmp is wiped with the VM, so a fresh session recompiles (~53 s); Drive survives.
CACHE_DIR = '/tmp/af3_cache'
if persist_cache_to_drive:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/.af3_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f'Compile cache: {CACHE_DIR} (survives this session)')
  except Exception as _e:
    print(f'(Drive mount failed, using {CACHE_DIR}: {_e})')


def _await(sentinel, limit=1200):
  """Wait for a background job, reporting its log if it never finishes."""
  t0 = time.time()
  while not os.path.isfile(sentinel):
    if time.time() - t0 > limit:
      log = f'{sentinel}.log'
      tail = open(log).read()[-1500:] if os.path.isfile(log) else '(no output captured)'
      raise RuntimeError(f'{sentinel} did not appear within {limit} s. '
                         f'Tail of {log}:\n{tail}')
    time.sleep(5)
  print(f'{sentinel} \u2713  ({time.time() - t0:.0f} s)')


_await(STAMP)

if IS_AF3 and os.path.getsize(f'{NATIVE_DIR}/af3.bin.zst') < 1_000_000:
  raise RuntimeError('the AlphaFold 3 download is incomplete - re-run this cell.')

print(f'Setup complete!  Model: {model}.')
if model == 'chai1':
  print('NOTE: chai-1 is running WITHOUT ESM2 embeddings, which are most of its token\n'
        '      features. Expect worse structures than chai-lab itself produces.')
print(f'Setup took {time.time() - _T0:.0f} s.')


overlaid the colab branch onto the 3.1.11 wheel
tokamax patch not needed on this device (gpu, cc 7.5).
WEIGHTS_DONE_openbind0_int8 ✓  (0 s)
Setup complete!  Model: openbind0.
Setup took 2 s.


In [ ]:
# The sm_75 BACKWARD, on the card it is for. It has only ever run on an sm86
# build of the same source; the shipped sm75 libraries have never executed on a
# T4, and the notebook installs the upstream wheel, which has no backward at all.
import subprocess, sys
WHEEL = ('https://github.com/sokrypton/colabfold-legacy-kernels/releases/download/'
         'volta-bwd-0.3.0.dev0/colabfold_legacy_kernels-0.3.0.dev0-py3-none-linux_x86_64.whl')
print(subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
                      '--no-deps', WHEEL], capture_output=True, text=True).returncode)
import colabfold_legacy_kernels as clk
print('version', clk.__version__)
print('attention     ', clk.library_path('attention', 75))
print('attention_bwd ', clk.library_path('attention_bwd', 75))


0
version 0.3.0.dev0
attention      /usr/local/lib/python3.13/dist-packages/colabfold_legacy_kernels/kernels/sm75/libvolta_mma.so
attention_bwd  /usr/local/lib/python3.13/dist-packages/colabfold_legacy_kernels/kernels/sm75/libvolta_mma_bwd.so


In [ ]:
# 1. the kernel gate, straight from the fork's own test
import subprocess, sys, os, urllib.request
os.makedirs('t', exist_ok=True)
url = ('https://raw.githubusercontent.com/sokrypton/colabfold-legacy-kernels/'
       'backward/tests/test_attn_bwd.py')
urllib.request.urlretrieve(url, 't/test_attn_bwd.py')
import colabfold_legacy_kernels as clk
kd = os.path.dirname(clk.library_path('attention', 75))
env = dict(os.environ, KERNEL_DIR=kd)
p = subprocess.run([sys.executable, 't/test_attn_bwd.py'], capture_output=True,
                   text=True, env=env)
print(p.stdout[-2500:])
print(p.stderr[-1500:] if p.returncode else '')


{'sq': 96, 'sk': 96, 'd': 32}
  out    rel 3.71e-04   max|ref| 0.269
  lse2   rel 1.38e-07   max|ref| 6.917
  dq     rel 4.15e-04   max|ref| 0.075
  dk     rel 4.30e-04   max|ref| 0.091
  dv     rel 5.07e-04   max|ref| 0.287
  dbias  rel 4.03e-05   max|ref| 0.292
  -> OK (worst 5.07e-04)
{'sq': 96, 'sk': 96, 'd': 32, 'bq': 64, 'bk': 64}
  out    rel 3.71e-04   max|ref| 0.269
  lse2   rel 1.38e-07   max|ref| 6.917
  dq     rel 4.15e-04   max|ref| 0.075
  dk     rel 4.30e-04   max|ref| 0.091
  dv     rel 5.07e-04   max|ref| 0.287
  dbias  rel 3.77e-05   max|ref| 0.292
  -> OK (worst 5.07e-04)
{'sq': 64, 'sk': 64, 'd': 64, 'bk': 64}
  out    rel 4.40e-04   max|ref| 0.351
  lse2   rel 1.51e-07   max|ref| 6.318
  dq     rel 3.98e-04   max|ref| 0.105
  dk     rel 3.75e-04   max|ref| 0.121
  dv     rel 3.72e-04   max|ref| 0.461
  dbias  rel 3.94e-05   max|ref| 0.754
  -> OK (worst 4.40e-04)
{'sq': 70, 'sk': 83, 'd': 16}
  out    rel 3.51e-04   max|ref| 0.268
  lse2   rel 1.42e-07   max|ref| 6

In [ ]:
# 2. the same thing through OUR wrapper, which is what a design run calls
import jax, jax.numpy as jnp, numpy as np
from alphafold3.model.components import volta_attn as va, platform

cc = va.device_cc()
print('cc', cc, '| installed', va.installed(), '| bwd_installed', va.bwd_installed(cc),
      '| bwd_available', va.bwd_available(cc))
print('differentiable row ->', platform.attention_config(differentiable=True)['attention'])

b, sq, h, c = 3, 96, 4, 32
k0 = jax.random.PRNGKey(0)
q, k, v = [jax.random.normal(jax.random.fold_in(k0, i), (b, sq, h, c), jnp.float32) * 0.5
           for i in range(3)]
bias = jax.random.normal(jax.random.fold_in(k0, 3), (1, h, sq, sq), jnp.float32) * 0.5
mask = (jax.random.uniform(jax.random.fold_in(k0, 4), (b, 1, 1, sq)) > 0.15)
scale = c ** -0.5
w = jnp.sin(jnp.arange(b * sq * h * c, dtype=jnp.float32)).reshape(b, sq, h, c)

def ref(q, k, v, bias):
  logits = scale * jnp.einsum('bqhc,bkhc->bhqk', q, k) + bias
  return jnp.einsum('bhqk,bkhc->bqhc',
                    jax.nn.softmax(jnp.where(mask, logits, -1e4), -1), v)

def kern(q, k, v, bias):
  out = va.attention(q.astype(jnp.float16), k.astype(jnp.float16), v.astype(jnp.float16),
                     mask=mask, bias=bias.astype(jnp.float16), scale=scale, cc=cc)
  assert out is not None, 'the volta arm refused'
  return out.astype(jnp.float32)

loss = lambda f: (lambda *a: jnp.sum(f(*a) * w))
g_ref = jax.grad(loss(ref), argnums=(0, 1, 2, 3))(q, k, v, bias)
g_got = jax.grad(loss(kern), argnums=(0, 1, 2, 3))(q, k, v, bias)
ok = True
for name, a, e in zip(('dq', 'dk', 'dv', 'dbias'), g_got, g_ref):
  a, e = np.asarray(a, np.float64), np.asarray(e, np.float64)
  rel = np.abs(a - e).max() / max(np.abs(e).max(), 1e-9)
  ok &= rel < 0.05
  print(f'  {name:6s} rel {rel:.2e}')
print('RESULT:', 'PASS' if ok else 'FAIL')


cc 75 | installed True | bwd_installed True | bwd_available False
differentiable row -> volta


ValueError: The FFI call to `VoltaMma` cannot be differentiated. You can use `jax.custom_jvp` or `jax.custom_jvp` to add support.